# 🌊 Smoke Run WAVE-CRE (Baseline B5 Comparator)
### Adaptive Prompting for Continual Relation Extraction (FewRel 10 tasks × 8 relations, Seed 2021)

Notebook này phục vụ việc thực thi kiểm chứng (smoke run) 1 seed duy nhất cho baseline **B5 (WAVE-CRE)** trên môi trường GPU của Google Colab.
- **Repository:** `research-code-lab` (Branch: `feat/b5-wave-smoke`)
- **Mã nguồn:** `paper-reproductions/WAVE-CRE` (Upstream commit: `5273b70`)
- **Tính năng nâng cao:** Hỗ trợ **Checkpoint sau mỗi task & Tự động Resume** nếu Colab bị ngắt kết nối giữa chừng.

## 1. Kiểm tra Môi trường GPU (NVIDIA CUDA Verification)

In [19]:
!git -C /content/research-code-lab pull origin feat/b5-wave-smoke

remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 6 (delta 5), reused 6 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 562 bytes | 281.00 KiB/s, done.
From https://github.com/vinhlq2512/research-code-lab
 * branch            feat/b5-wave-smoke -> FETCH_HEAD
   fb93e75..754b4fa  feat/b5-wave-smoke -> origin/feat/b5-wave-smoke
Updating fb93e75..754b4fa
Fast-forward
 paper-reproductions/WAVE-CRE/methods/manager.py | 5 ++++-
 1 file changed, 4 insertions(+), 1 deletion(-)


In [20]:
import torch
import os
import sys

print("=" * 60)
print("CUDA / GPU PREFLIGHT CHECK")
print("=" * 60)
if torch.cuda.is_available():
    print(f"✓ CUDA Available: True")
    print(f"✓ GPU Model:     {torch.cuda.get_device_name(0)}")
    print(f"✓ VRAM:          {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    !nvidia-smi
else:
    print("⚠️ CẢNH BÁO: CUDA không khả dụng! Hãy vào Runtime -> Change runtime type -> Chọn T4 GPU.")

CUDA / GPU PREFLIGHT CHECK
✓ CUDA Available: True
✓ GPU Model:     Tesla T4
✓ VRAM:          15.64 GB
Sun Nov 15 06:49:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                     

## 2. (Khuyến nghị) Kết nối Google Drive để Bảo Toàn Checkpoints & Auto-Resume
Mount Google Drive để lưu checkpoints trực tiếp sau mỗi task. Nếu Colab bị rớt mạng giữa chừng, lần chạy sau sẽ tự động nối tiếp từ checkpoint này.

In [21]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

GDRIVE_BACKUP_DIR = Path('/content/drive/MyDrive/research-code-lab-runs/wave_cre/seed_2021')
GDRIVE_CHECKPOINTS_DIR = GDRIVE_BACKUP_DIR / 'checkpoints'
GDRIVE_CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"✓ Đã kết nối Google Drive!")
print(f"  - Thư mục sao lưu:   {GDRIVE_BACKUP_DIR}")
print(f"  - Thư mục checkpoint: {GDRIVE_CHECKPOINTS_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Đã kết nối Google Drive!
  - Thư mục sao lưu:   /content/drive/MyDrive/research-code-lab-runs/wave_cre/seed_2021
  - Thư mục checkpoint: /content/drive/MyDrive/research-code-lab-runs/wave_cre/seed_2021/checkpoints


## 3. Clone Repository research-code-lab & Cài Đặt Môi Trường

In [22]:
from pathlib import Path

WORKDIR = Path("/content")
%cd {WORKDIR}

# Clone repo chính thức trên nhánh feat/b5-wave-smoke
!git clone -b feat/b5-wave-smoke https://github.com/vinhlq2512/research-code-lab.git

REPO_DIR = WORKDIR / "research-code-lab"
WAVE_DIR = REPO_DIR / "paper-reproductions/WAVE-CRE"
%cd {WAVE_DIR}

# Cài đặt dependencies chính xác cho WAVE-CRE
!pip install -q transformers gpytorch linear_operator python-dotenv pyyaml

# Khởi tạo thư mục artifact theo quy chuẩn Protocol v1
RUN_DIR = REPO_DIR / "runs/wave_cre/fewrel_author_config/seed_2021"
RUN_DIR.mkdir(parents=True, exist_ok=True)

# Lưu thông tin môi trường để bảo đảm reproducibility
!pip freeze > {RUN_DIR}/env_info.txt
!nvidia-smi > {RUN_DIR}/gpu_info.txt
print(f"✓ Môi trường đã sẵn sàng!")

/content
fatal: destination path 'research-code-lab' already exists and is not an empty directory.
/content/research-code-lab/paper-reproductions/WAVE-CRE
✓ Môi trường đã sẵn sàng!


## 4. Chạy Thực Nghiệm Smoke 1 Seed (Hỗ trợ Checkpoint & Auto-Resume)

In [23]:
import subprocess
import sys
from pathlib import Path

%cd {WAVE_DIR}

LOG_FILE = RUN_DIR / "run.log"

# Cấu hình thư mục lưu checkpoint (ưu tiên Google Drive nếu có kết nối)
if Path('/content/drive/MyDrive').exists():
    CKPT_DIR = GDRIVE_CHECKPOINTS_DIR
else:
    CKPT_DIR = RUN_DIR / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# Kiểm tra xem đã có checkpoint trước đó để tự động resume hay không
latest_ckpt = CKPT_DIR / "FewRel_2021_latest.pt"
resume_flag = "--resume " if latest_ckpt.exists() else ""

if resume_flag:
    print(f"🔄 PHÁT HIỆN CHECKPOINT TRƯỚC ĐÓ: {latest_ckpt}")
    print(f"   -> Tự động kích hoạt --resume để tiếp tục từ task kế tiếp!")
else:
    print(f"🚀 BẮT ĐẦU HUẤN LUYỆN TỪ TASK 1... Checkpoint lưu tại: {CKPT_DIR}")

cmd = (
    "python -u run.py "
    "--max_length 256 "
    "--dataname FewRel "
    "--encoder_epochs 30 "
    "--encoder_lr 2e-5 "
    "--prompt_pool_epochs 25 "
    "--prompt_pool_lr 1e-4 "
    "--classifier_epochs 250 "
    "--seed 2021 "
    "--bert_path bert-base-uncased "
    "--data_path datasets "
    "--prompt_length 8 "
    "--prompt_top_k 4 "
    "--batch_size 16 "
    "--prompt_pool_size 20 "
    "--replay_s_e_e 100 "
    "--replay_epochs 200 "
    "--classifier_lr 5e-5 "
    "--prompt_type only_prompt "
    f"--checkpoint_dir {CKPT_DIR} "
    f"{resume_flag}"
)

# Chạy và stream log ra cell output đồng thời ghi vào file run.log
mode = "a" if resume_flag else "w"
with open(LOG_FILE, mode) as f:
    process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        sys.stdout.write(line)
        f.write(line)
    process.wait()

if process.returncode == 0:
    print("\n✅ Thực nghiệm hoàn thành thành công!")
else:
    print(f"\n❌ Lỗi thực thi! Mã thoát: {process.returncode}")

Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
train_prompted_classifier_epoch_237: 100%|██████████| 250/250 [00:00<00:00, 379.53it/s, acc=1, loss=0.00197]

train_prompted_classifier_epoch_238: 100%|██████████| 250/250 [00:00<00:00, 374.15it/s, acc=0.999, loss=0.00338]

train_prompted_classifier_epoch_239: 100%|██████████| 250/250 [00:00<00:00, 368.79it/s, acc=0.999, loss=0.00437]

train_prompted_classifier_epoch_240: 100%|██████████| 250/250 [00:00<00:00, 363.90it/s, acc=0.997, loss=0.00997]

train_prompted_classifier_epoch_241: 100%|██████████| 250/250 [00:00<00:00, 370.70it/s, acc=0.998, loss=0.00774]

train_prompted_classifier_epoch_242: 100%|██████████| 250/250 [00:00<00:00, 370.37it/s, acc=0.997, loss=0.00546]

train_prompted_classifier_epoch_243: 100%|██████████| 250/250 [00:00<00:00, 358.59it/s, acc=0.999, loss=0.00318]

train_prompted_classifier_epoch_244: 100%|██████████| 250/250 [00:00<00:00, 362.25it/s, acc=0.999, loss=0.00254]

train_prompted_classifier_epoch_245

## 5. Trích Xuất & Chuẩn Hóa Metrics Sang Chuẩn Protocol v1

In [24]:
PARSE_SCRIPT = REPO_DIR / "scripts/parse_wave_results.py"
RESULTS_DIR = WAVE_DIR / "results/2021_coda_prompt"
OUTPUT_METRICS = RUN_DIR / "metrics.json"

!python {PARSE_SCRIPT} {RESULTS_DIR} --out {OUTPUT_METRICS} --meta seed=2021 dataset=FewRel model=WAVE-CRE

import json
if OUTPUT_METRICS.exists():
    with open(OUTPUT_METRICS, "r") as f:
        m = json.load(f)
    print("\n--- KẾT QUẢ THỰC NGHIỆM CHÍNH THỨC B5 (SWAG PREDICTED POOL) ---")
    print(json.dumps(m["derived_swag_predicted_pool"], indent=2))
else:
    print("⚠️ Chưa tìm thấy metrics.json. Kiểm tra lại log chạy.")

tasks=5 final_acc_whole=0.9168 forgetting=0.03504464285714287 oracle_gap=0.0000

--- KẾT QUẢ THỰC NGHIỆM CHÍNH THỨC B5 (SWAG PREDICTED POOL) ---
{
  "accuracy_whole_per_step": [
    0.9866071428571429,
    0.9352678571428571,
    0.9398809523809524,
    0.9285714285714286,
    0.9167857142857143
  ],
  "accuracy_average_per_step": [
    0.9866071428571429,
    0.9352678571428571,
    0.9398809523809524,
    0.9285714285714286,
    0.9167857142857143
  ],
  "final_accuracy_whole": 0.9167857142857143,
  "average_incremental_accuracy_whole": 0.941422619047619,
  "average_forgetting": 0.03504464285714287,
  "bwt": -0.03504464285714287,
  "final_oracle_minus_predicted_gap": 0.0
}


## 6. Sao Lưu Toàn Bộ Artifacts & Logs Lên Google Drive

In [25]:
import shutil

if Path('/content/drive/MyDrive').exists():
    print(f"Đang đồng bộ từ {RUN_DIR} sang {GDRIVE_BACKUP_DIR}...")
    shutil.copytree(RUN_DIR, GDRIVE_BACKUP_DIR, dirs_exist_ok=True)
    print(f"✅ Đã sao lưu toàn bộ artifacts và logs thành công tại:")
    print(f"   {GDRIVE_BACKUP_DIR}")
else:
    print("⚠️ Google Drive chưa được mount. Hãy download thủ công thư mục runs trên Colab.")

Đang đồng bộ từ /content/research-code-lab/runs/wave_cre/fewrel_author_config/seed_2021 sang /content/drive/MyDrive/research-code-lab-runs/wave_cre/seed_2021...
✅ Đã sao lưu toàn bộ artifacts và logs thành công tại:
   /content/drive/MyDrive/research-code-lab-runs/wave_cre/seed_2021
